# Machine Learning in Cybersecurity: Threat Detection and Response
### Real-data implementation based on RP1 — Gadad, Varshitha, Yoghana & Yumlembam Singh (ICAET 2025)

This notebook turns the ideas in the paper **“Machine Learning in Cybersecurity: Threat Detection and Response”** into a runnable implementation using the **CICIDS2017** network-traffic dataset.

### Implementation idea

- **Supervised learning:** learn to distinguish normal traffic from **known attacks**.
- **Unsupervised learning:** train an **Isolation Forest on normal traffic only** and test it against an attack family that was completely withheld from supervised training.
- **Hybrid detection:** use the supervised model for known attacks and the anomaly detector as a safety net for unseen traffic patterns.
- **Automated response:** map the final verdict to a simple incident-response action.

> This is an educational implementation of the concepts discussed in the paper, not a reproduction of the paper's experimental results.

## 0. Before running this notebook

The CICIDS2017 CSV files should already be available in:

```text
/content/cicids2017/
```


In [1]:
import os
import glob
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

print('Python ML environment ready.')

Python ML environment ready.


## 1. Locate and load CICIDS2017

The original dataset contains multiple CSV files. To keep the implementation practical on a free Colab runtime, we combine the files and then take a **stratified sample**. Stratification keeps the proportions of the original labels approximately intact.

In [ ]:
DATA_DIR_CANDIDATES = [
    './cicids2017',
    '/content/cicids2017',
    os.path.abspath('./cicids2017'),
    os.path.abspath('/content/cicids2017')
]

DATA_DIR = next((p for p in DATA_DIR_CANDIDATES if os.path.isdir(p)), './cicids2017')
SAMPLE_SIZE = 250_000

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        'Could not find the CICIDS2017 CSV folder. Download or upload the dataset into one of these locations:\n'
        + '\n'.join(DATA_DIR_CANDIDATES)
        + '\n\nIf you are in Colab, run the data-download notebook first. If you are on a local machine, '
        'make sure the extracted CSV files are present under the selected folder.'
    )

print(f'Using dataset directory: {DATA_DIR}')

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, '**', '*.csv'), recursive=True))
if not csv_files:
    raise FileNotFoundError(f'No CSV files found under {DATA_DIR}')

print(f'Found {len(csv_files)} CSV files.')
for f in csv_files:
    print(f' - {os.path.basename(f)}')

FileNotFoundError: Could not find ./cicids2017. Run step2_data_collection.ipynb first, or upload/extract the CICIDS2017 CSV files into /content/cicids2017.

In [ ]:
def optimise_dtypes(df):
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df

parts = []
for f in csv_files:
    part = pd.read_csv(f, low_memory=False)
    part.columns = part.columns.str.strip()
    part['source_file'] = os.path.basename(f)
    parts.append(optimise_dtypes(part))
    print(f'Loaded {os.path.basename(f):55s} -> {part.shape}')

df = pd.concat(parts, ignore_index=True)
del parts

print(f'Combined shape: {df.shape}')

In [ ]:
# Clean label text and inspect the original attack classes.
if 'Label' not in df.columns:
    raise KeyError("Expected a 'Label' column in CICIDS2017.")

df['Label'] = df['Label'].astype(str).str.strip()

label_counts = df['Label'].value_counts()
print('Original label distribution:')
display(label_counts.to_frame('count'))

In [ ]:
# Stratified sampling for a practical Colab run.
if SAMPLE_SIZE and len(df) > SAMPLE_SIZE:
    sample = (
        df.groupby('Label', group_keys=False)
          .apply(lambda g: g.sample(
              n=min(len(g), max(1, int(round(SAMPLE_SIZE * len(g) / len(df))))),
              random_state=RANDOM_SEED
          ))
          .reset_index(drop=True)
    )
else:
    sample = df.copy()

del df

print(f'Sampled dataset shape: {sample.shape}')
print('Sampled labels:')
display(sample['Label'].value_counts().to_frame('count'))

## 2. Define the threat-detection task

To make the supervised problem stable despite the many rare CICIDS2017 attack labels, we use a **binary known-threat classifier**:

- `normal` → BENIGN traffic
- `known_attack` → every attack type **except Bot**
- `novel` → **Bot** traffic, completely withheld from supervised training

This makes the experiment easy to explain: *Can supervised ML detect attacks it has seen before, and can an anomaly detector flag an unseen attack family?*

In [ ]:
def map_security_category(label):
    s = label.lower()
    if s == 'benign':
        return 'normal'
    if 'bot' in s:
        return 'novel'
    return 'known_attack'

sample['security_category'] = sample['Label'].map(map_security_category)

print(sample['security_category'].value_counts())

novel_count = int((sample['security_category'] == 'novel').sum())
if novel_count < 50:
    raise ValueError(
        f'Only {novel_count} Bot samples were found. Increase SAMPLE_SIZE or use a larger dataset sample.'
    )

In [ ]:
# Keep the original label for interpretation, but exclude it from ML features.
known_df = sample[sample['security_category'] != 'novel'].copy()
novel_df = sample[sample['security_category'] == 'novel'].copy()

print('Known-data shape  :', known_df.shape)
print('Novel-data shape  :', novel_df.shape)
print('Novel attack label:', novel_df['Label'].value_counts().to_dict())

## 3. Prepare numeric network-flow features

CICIDS2017 contains numeric flow statistics such as packet counts, packet lengths, inter-arrival times, flags, rates and window sizes. Metadata (`Label`, `security_category`, and `source_file`) are excluded.

The preprocessing steps are:

1. Convert `inf`/`-inf` to missing values.
2. Keep numeric traffic features.
3. Remove columns that are entirely missing or constant.
4. Split training/test data.
5. Impute missing values and standardize the features using **training data only**.

In [ ]:
meta_cols = {'Label', 'security_category', 'source_file'}
feature_candidates = [c for c in known_df.columns if c not in meta_cols]

# Convert candidate features to numeric and remove non-numeric metadata.
X_all = known_df[feature_candidates].apply(pd.to_numeric, errors='coerce')
X_novel_raw = novel_df[feature_candidates].apply(pd.to_numeric, errors='coerce')

# Replace infinities generated by rate features.
X_all = X_all.replace([np.inf, -np.inf], np.nan)
X_novel_raw = X_novel_raw.replace([np.inf, -np.inf], np.nan)

# Keep features with at least some observed values.
valid_cols = X_all.columns[X_all.notna().any()].tolist()
X_all = X_all[valid_cols]
X_novel_raw = X_novel_raw[valid_cols]

# Binary supervised target: normal vs known attack.
y_all = known_df['security_category'].values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y_all
)

print('Initial numeric feature count:', len(valid_cols))
print('Training rows:', len(X_train_raw))
print('Test rows    :', len(X_test_raw))

In [ ]:
# Fit preprocessing only on the training set.
preprocess = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('variance', VarianceThreshold(threshold=0.0)),
    ('scaler', StandardScaler())
])

X_train = preprocess.fit_transform(X_train_raw)
X_test = preprocess.transform(X_test_raw)
X_novel = preprocess.transform(X_novel_raw)

print('Processed feature count:', X_train.shape[1])
print('X_train shape:', X_train.shape)
print('X_test shape :', X_test.shape)
print('X_novel shape:', X_novel.shape)

## 4. Supervised learning — known-threat classification

The paper discusses supervised techniques such as decision trees, SVMs and neural networks, as well as ensemble methods. We compare four practical models:

- Decision Tree
- **Linear SVM** (a scalable SVM variant for large tabular data)
- MLP neural network
- Random Forest

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(
        max_depth=12,
        class_weight='balanced',
        random_state=RANDOM_SEED
    ),
    'Linear SVM': LinearSVC(
        class_weight='balanced',
        random_state=RANDOM_SEED,
        dual=False,
        max_iter=3000
    ),
    'MLP Neural Network': MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation='relu',
        early_stopping=True,
        validation_fraction=0.1,
        batch_size=512,
        max_iter=40,
        random_state=RANDOM_SEED
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=150,
        class_weight='balanced_subsample',
        n_jobs=-1,
        random_state=RANDOM_SEED
    )
}

metrics_rows = []
trained_models = {}

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    trained_models[name] = model
    metrics_rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, pos_label='known_attack'),
        'Recall': recall_score(y_test, preds, pos_label='known_attack'),
        'F1': f1_score(y_test, preds, pos_label='known_attack'),
        'Train time (s)': round(time.time() - t0, 1)
    })

results_supervised = pd.DataFrame(metrics_rows).sort_values('F1', ascending=False)
display(results_supervised)

In [ ]:
rf = trained_models['Random Forest']
rf_test_pred = rf.predict(X_test)

print('Random Forest classification report:')
print(classification_report(y_test, rf_test_pred, digits=4))

cm = confusion_matrix(y_test, rf_test_pred, labels=['normal', 'known_attack'])
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Normal', 'Known attack']
).plot(values_format='d')
plt.title('Random Forest — Known-threat classification')
plt.show()

## 5. Unsupervised learning — detecting an unseen attack family

The supervised models have **never seen Bot traffic** during training. To model the paper's discussion of unknown/zero-day threats, the anomaly detector is trained only on **normal training traffic**.

We use **Isolation Forest** because it is practical for a large tabular dataset. The anomaly detector does not receive attack labels during training.

In [ ]:
normal_train_mask = (y_train == 'normal')
X_normal_train = X_train[normal_train_mask]

# Cap the fitting sample for faster execution in Colab.
max_iforest_samples = 20_000
if len(X_normal_train) > max_iforest_samples:
    idx = rng.choice(len(X_normal_train), size=max_iforest_samples, replace=False)
    X_normal_fit = X_normal_train[idx]
else:
    X_normal_fit = X_normal_train

iso = IsolationForest(
    n_estimators=150,
    max_samples='auto',
    contamination=0.03,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

t0 = time.time()
iso.fit(X_normal_fit)
print(f'Isolation Forest trained on {len(X_normal_fit):,} normal samples in {time.time() - t0:.1f}s')

In [ ]:
X_test_normal = X_test[y_test == 'normal']

iso_novel_pred = iso.predict(X_novel)       # -1 = anomaly
iso_normal_pred = iso.predict(X_test_normal)

novel_catch = float((iso_novel_pred == -1).mean())
normal_false_alarm = float((iso_normal_pred == -1).mean())
rf_novel_pred = rf.predict(X_novel)
rf_novel_miss = float((rf_novel_pred == 'normal').mean())

unsupervised_results = pd.DataFrame({
    'Measure': [
        'Isolation Forest novel-attack catch rate',
        'Isolation Forest false-alarm rate on normal traffic',
        'Random Forest novel-attack miss rate'
    ],
    'Rate': [novel_catch, normal_false_alarm, rf_novel_miss]
})

display(unsupervised_results)

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.bar(
    ['IF catch\nnovel Bot', 'IF false alarms\nnormal', 'RF misses\nnovel Bot'],
    [novel_catch, normal_false_alarm, rf_novel_miss]
)
plt.ylim(0, 1.05)
plt.ylabel('Rate')
plt.title('Known-threat model vs. anomaly detector on unseen Bot traffic')
plt.show()

## 6. Hybrid threat detector

The hybrid rule is:

1. If the Random Forest says **known_attack**, keep that result.
2. If it says **normal**, send the traffic through Isolation Forest.
3. If Isolation Forest flags an anomaly, output **unknown_suspected**.
4. Otherwise output **normal**.

This creates the supervised + unsupervised safety net described in the paper.

In [ ]:
def hybrid_predict(X_scaled, supervised_model, anomaly_model):
    supervised_pred = supervised_model.predict(X_scaled)
    anomaly_pred = anomaly_model.predict(X_scaled)

    final = []
    for sup, anom in zip(supervised_pred, anomaly_pred):
        if sup == 'known_attack':
            final.append('known_attack')
        elif anom == -1:
            final.append('unknown_suspected')
        else:
            final.append('normal')
    return np.array(final)

# Evaluate on known test traffic + completely unseen Bot traffic.
X_eval = np.vstack([X_test, X_novel])
y_eval_true = np.concatenate([y_test, np.array(['novel'] * len(X_novel))])

baseline_pred = rf.predict(X_eval)
hybrid_pred = hybrid_predict(X_eval, rf, iso)

known_attack_mask = y_eval_true == 'known_attack'
novel_mask = y_eval_true == 'novel'
normal_mask = y_eval_true == 'normal'

baseline_known_recall = (baseline_pred[known_attack_mask] == 'known_attack').mean()
baseline_novel_catch = (baseline_pred[novel_mask] != 'normal').mean()
hybrid_known_recall = (hybrid_pred[known_attack_mask] == 'known_attack').mean()
hybrid_novel_catch = (hybrid_pred[novel_mask] != 'normal').mean()

baseline_false_alarm = (baseline_pred[normal_mask] != 'normal').mean()
hybrid_false_alarm = (hybrid_pred[normal_mask] != 'normal').mean()

hybrid_summary = pd.DataFrame({
    'Metric': [
        'Known attack recall',
        'Unseen Bot catch rate',
        'False-alarm rate on normal traffic'
    ],
    'Random Forest only': [baseline_known_recall, baseline_novel_catch, baseline_false_alarm],
    'Hybrid RF + Isolation Forest': [hybrid_known_recall, hybrid_novel_catch, hybrid_false_alarm]
})

display(hybrid_summary)

In [ ]:
plot_df = hybrid_summary.set_index('Metric')
ax = plot_df.plot(kind='bar', figsize=(9, 5))
ax.set_ylim(0, 1.05)
ax.set_ylabel('Rate')
ax.set_title('Hybrid detection compared with supervised-only detection')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

## 7. Automated incident-response layer

This is intentionally a **simulation**, not a real firewall/SOAR integration. The goal is to demonstrate the final step from detection to a response action.

In [ ]:
RESPONSE_MAP = {
    'normal': 'No automated action. Continue normal monitoring.',
    'known_attack': 'Raise security alert and apply attack-specific investigation/mitigation.',
    'unknown_suspected': 'Quarantine or isolate the source/host for analyst review.'
}

# Show a mixed sample, including unseen Bot traffic.
sample_idx = rng.choice(len(y_eval_true), size=min(12, len(y_eval_true)), replace=False)

response_table = pd.DataFrame({
    'True category': y_eval_true[sample_idx],
    'Detector verdict': hybrid_pred[sample_idx],
    'Automated response': [RESPONSE_MAP[v] for v in hybrid_pred[sample_idx]]
})

display(response_table)

## 8. Final observations

- **Supervised models** learn the normal-vs-known-attack boundary from labeled CICIDS2017 traffic.
- **Isolation Forest** is trained only on normal traffic and provides a mechanism for flagging an attack family that the supervised model never saw.
- The **hybrid detector** combines both signals: known attacks are classified directly, while suspicious traffic that looks normal to the supervised model can still be escalated as `unknown_suspected`.
- The **response layer** demonstrates how a detection verdict can be connected to an incident-response action.

### Important limitation

CICIDS2017 is a benchmark dataset. Results obtained here should **not** be interpreted as proof that the detector will perform equally well on live production network traffic. The notebook is an academic implementation of the concepts from RP1.

In [ ]:
# Optional: save the fitted supervised model and preprocessing pipeline for the project demo.
import joblib

joblib.dump(rf, 'rf_cicids2017_model.joblib')
joblib.dump(preprocess, 'cicids2017_preprocess.joblib')
joblib.dump(iso, 'isolation_forest_cicids2017.joblib')

print('Saved:')
print(' - rf_cicids2017_model.joblib')
print(' - cicids2017_preprocess.joblib')
print(' - isolation_forest_cicids2017.joblib')